# Project 2 — Healthcare Claims Data Quality & Anomaly Detection

**Tools:** SQLite / SQL • Python / Pandas / NumPy • Power BI

This notebook uses the supplied synthetic healthcare claims dataset. The goal is to validate the claims, understand utilization and financial patterns, identify unusual operational patterns, and prepare Power BI-ready reporting.

> **Important:** An anomaly flag is a screening signal for review. It is not proof of fraud or misconduct.

## Roadmap

`Raw data → SQL quality checks → SQL business analysis → Python enrichment → anomaly detection → evaluation → Power BI`

The notebook uses small, readable cells so that every step can be explained line-by-line in an interview.

In [1]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
PROJECT_ROOT = Path("/mnt/data/innovaccer_project2_claims_anomaly_project")
DATABASE_PATH = PROJECT_ROOT / "01_data_raw" / "innovaccer_claims.db"
OUTPUT_PATH = PROJECT_ROOT / "05_outputs"

print(DATABASE_PATH)

/mnt/data/innovaccer_project2_claims_anomaly_project/01_data_raw/innovaccer_claims.db


## 1. Connect to SQLite and inspect the tables

In [3]:
connection = sqlite3.connect(DATABASE_PATH)

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection,
)

tables

,name
0,claim_lines
1,claims
2,diagnosis_catalog
3,facilities
4,patients
5,procedure_catalog
6,providers


In [4]:
patients = pd.read_sql_query("SELECT * FROM patients", connection)
providers = pd.read_sql_query("SELECT * FROM providers", connection)
facilities = pd.read_sql_query("SELECT * FROM facilities", connection)
claims = pd.read_sql_query("SELECT * FROM claims", connection)
claim_lines = pd.read_sql_query("SELECT * FROM claim_lines", connection)
diagnosis = pd.read_sql_query("SELECT * FROM diagnosis_catalog", connection)
procedure = pd.read_sql_query("SELECT * FROM procedure_catalog", connection)
truth = pd.read_csv(PROJECT_ROOT / "01_data_raw" / "anomaly_ground_truth.csv")

In [5]:
pd.DataFrame({
    "Table": [
        "patients", "providers", "facilities", "claims",
        "claim_lines", "diagnosis_catalog", "procedure_catalog"
    ],
    "Rows": [
        len(patients), len(providers), len(facilities), len(claims),
        len(claim_lines), len(diagnosis), len(procedure)
    ]
})

,Table,Rows
0,patients,20000
1,providers,450
2,facilities,25
3,claims,75025
4,claim_lines,123343
5,diagnosis_catalog,25
6,procedure_catalog,25


## 2. Inspect and type the claims data

In [6]:
claims.head(10)

,Claim_ID,Patient_ID,Service_Date,Submission_Date,Provider_ID,Facility_ID,Payer,Claim_Type,Claim_Status,Diagnosis_Code,Primary_Procedure_Code,Claim_Amount,Allowed_Amount,Paid_Amount,Member_Responsibility,Quality_Issue
0,C00000001,P003946,2024-10-28,2024-11-02,DR0048,F013,Medicare Advantage,Professional,Paid,DX011,PR001,43.04,31.71,30.46,0.21,
1,C00000002,P003748,2024-01-18,2024-01-20,DR0282,F011,Medicare Advantage,Professional,Paid,DX002,PR001,116.05,107.50,85.29,2.92,
2,C00000003,P017088,2025-08-08,2025-08-20,DR0029,F022,Workers Comp,Outpatient,Paid,DX025,PR007,35.30,25.04,22.38,0.16,
3,C00000004,P007229,2024-05-30,2024-06-03,DR0115,F019,Medicare,Professional,Paid,DX025,PR025,93.36,72.98,70.13,0.37,
4,C00000005,P017513,2024-09-18,2024-09-21,,F019,Medicaid,Outpatient,Reversed,DX007,PR012,406.32,364.03,67.66,8.12,Missing Provider
5,C00000006,P009969,2026-04-05,2026-04-11,DR0111,F020,Medicare,Professional,Pending,DX003,PR003,32.09,23.40,0.00,2.93,
6,C00000007,P013421,2024-03-05,2024-03-09,DR0416,F014,Medicare Advantage,Professional,Paid,DX005,PR015,"2,835.48","2,400.93","2,182.59",31.31,
7,C00000008,P011144,2025-01-06,2025-01-29,DR0035,F008,Medicaid,Professional,Paid,DX007,PR004,565.13,527.50,396.30,3.04,
8,C00000009,P005379,2024-05-20,2024-06-06,DR0121,F017,Medicare Advantage,Outpatient,Paid,DX022,PR001,106.41,91.86,76.51,2.15,
9,C00000010,P019904,2025-11-15,2025-11-25,DR0016,F020,Medicare,Professional,Paid,DX020,PR004,851.73,649.58,459.63,28.07,


In [7]:
claims.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75025 entries, 0 to 75024
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Claim_ID                75025 non-null  object 
 1   Patient_ID              75025 non-null  object 
 2   Service_Date            75025 non-null  object 
 3   Submission_Date         75025 non-null  object 
 4   Provider_ID             75025 non-null  object 
 5   Facility_ID             75025 non-null  object 
 6   Payer                   75025 non-null  object 
 7   Claim_Type              75025 non-null  object 
 8   Claim_Status            75025 non-null  object 
 9   Diagnosis_Code          75025 non-null  object 
 10  Primary_Procedure_Code  75025 non-null  object 
 11  Claim_Amount            75025 non-null  float64
 12  Allowed_Amount          75025 non-null  float64
 13  Paid_Amount             75025 non-null  float64
 14  Member_Responsibility   75025 non-null

In [8]:
claims["Service_Date"] = pd.to_datetime(claims["Service_Date"], errors="coerce")
claims["Submission_Date"] = pd.to_datetime(claims["Submission_Date"], errors="coerce")

In [9]:
financial_columns = [
    "Claim_Amount",
    "Allowed_Amount",
    "Paid_Amount",
    "Member_Responsibility",
]

for column in financial_columns:
    claims[column] = pd.to_numeric(
        claims[column],
        errors="coerce"
    )

## 3. Data-quality checks

In [10]:
duplicate_claim_rows = claims["Claim_ID"].duplicated(keep=False).sum()
missing_provider = claims["Provider_ID"].fillna("").eq("").sum()
missing_line_diagnosis = claim_lines["Diagnosis_Code"].fillna("").eq("").sum()
paid_gt_allowed = (claims["Paid_Amount"] > claims["Allowed_Amount"]).sum()
submission_before_service = (claims["Submission_Date"] < claims["Service_Date"]).sum()

print("Duplicate claim rows:", duplicate_claim_rows)
print("Missing provider IDs:", missing_provider)
print("Missing line diagnoses:", missing_line_diagnosis)
print("Paid > allowed:", paid_gt_allowed)
print("Submission before service:", submission_before_service)

Duplicate claim rows: 50
Missing provider IDs: 144
Missing line diagnoses: 91
Paid > allowed: 41
Submission before service: 28


## 4. Build the enriched fact table

In [11]:
fact = (
    claims
    .merge(
        patients[["Patient_ID", "Age", "Gender", "Region"]],
        on="Patient_ID",
        how="left"
    )
    .merge(
        providers[["Provider_ID", "Provider_Display_Name", "Specialty"]],
        on="Provider_ID",
        how="left"
    )
    .merge(
        facilities[["Facility_ID", "Facility_Name", "Facility_Type", "Facility_Size"]],
        on="Facility_ID",
        how="left"
    )
    .merge(
        diagnosis[["Diagnosis_Code", "Diagnosis_Name", "Clinical_Group"]],
        on="Diagnosis_Code",
        how="left"
    )
    .merge(
        procedure[["Procedure_Code", "Procedure_Name", "Procedure_Group"]],
        left_on="Primary_Procedure_Code",
        right_on="Procedure_Code",
        how="left"
    )
)

fact.shape

(75025, 29)

In [12]:
fact["Submission_Lag_Days"] = (
    fact["Submission_Date"] - fact["Service_Date"]
).dt.days

fact["Month"] = (
    fact["Service_Date"]
    .dt.to_period("M")
    .astype(str)
)

fact["Week_Start"] = (
    fact["Service_Date"]
    - pd.to_timedelta(
        fact["Service_Date"].dt.weekday,
        unit="D"
    )
).dt.date.astype(str)

fact["Paid_Rate"] = (
    fact["Paid_Amount"]
    / fact["Allowed_Amount"].replace(0, np.nan)
)

## 5. Descriptive analysis: payer performance

In [13]:
payer_summary = (
    fact
    .groupby("Payer", as_index=False)
    .agg(
        Claims=("Claim_ID", "count"),
        Patients=("Patient_ID", "nunique"),
        Billed=("Claim_Amount", "sum"),
        Allowed=("Allowed_Amount", "sum"),
        Paid=("Paid_Amount", "sum"),
        Denied_Rejected=(
            "Claim_Status",
            lambda values: values.isin(["Denied", "Rejected"]).sum()
        ),
    )
)

payer_summary["Payment_Rate"] = payer_summary["Paid"] / payer_summary["Allowed"]
payer_summary["Denial_Rejection_Rate"] = payer_summary["Denied_Rejected"] / payer_summary["Claims"]

payer_summary.sort_values("Claims", ascending=False)

,Payer,Claims,Patients,Billed,Allowed,Paid,Denied_Rejected,Payment_Rate,Denial_Rejection_Rate
2,Medicare,23314,5717,"13,691,635.36","11,451,463.54","7,741,563.75",3723,0.68,0.16
0,Commercial,21438,5331,"12,209,088.55","10,179,447.44","7,035,904.65",3333,0.69,0.16
1,Medicaid,12179,2957,"7,016,923.78","5,861,853.42","3,953,991.24",1991,0.67,0.16
3,Medicare Advantage,9987,2437,"5,934,097.75","4,958,855.56","3,298,033.04",1583,0.67,0.16
5,Self-Pay,5150,1264,"2,990,003.62","2,502,663.98","1,723,262.59",863,0.69,0.17
4,Other,1563,385,"1,045,091.11","856,857.18","540,171.27",245,0.63,0.16
6,Workers Comp,1394,375,"765,556.21","641,781.99","426,527.34",212,0.66,0.15


## 6. Descriptive analysis: specialty and provider volume

In [14]:
specialty_summary = (
    fact
    .groupby("Specialty", as_index=False)
    .agg(
        Claims=("Claim_ID", "count"),
        Patients=("Patient_ID", "nunique"),
        Avg_Claim=("Claim_Amount", "mean"),
        Billed=("Claim_Amount", "sum"),
        Paid=("Paid_Amount", "sum"),
    )
    .sort_values("Billed", ascending=False)
)

specialty_summary

,Specialty,Claims,Patients,Avg_Claim,Billed,Paid
9,Other,8594,6627,865.06,"7,434,307.51","4,250,571.44"
8,Orthopedics,5755,4780,"1,280.15","7,367,281.20","4,159,945.65"
3,General Surgery,2789,2523,"2,507.42","6,993,189.79","3,910,550.28"
1,Emergency Medicine,9699,7232,423.40,"4,106,519.68","2,351,628.67"
2,Family Medicine,13135,8861,236.96,"3,112,512.30","1,785,640.34"
4,Internal Medicine,11995,8441,241.04,"2,891,316.65","1,620,399.15"
0,Cardiology,5632,4650,457.40,"2,576,075.51","1,467,724.65"
7,Oncology,3560,3178,714.92,"2,545,123.14","1,446,273.26"
6,Obstetrics & Gynecology,2676,2454,847.22,"2,267,162.39","1,276,769.96"
11,Radiology,2754,2535,602.86,"1,660,265.73","936,073.60"


In [15]:
provider_summary = (
    fact[fact["Provider_ID"].fillna("").ne("")]
    .groupby(["Provider_ID", "Provider_Display_Name", "Specialty"], as_index=False)
    .agg(
        Claims=("Claim_ID", "count"),
        Patients=("Patient_ID", "nunique"),
        Avg_Claim=("Claim_Amount", "mean"),
        Billed=("Claim_Amount", "sum"),
        Paid=("Paid_Amount", "sum"),
    )
    .sort_values("Claims", ascending=False)
)

provider_summary.head(20)

,Provider_ID,Provider_Display_Name,Specialty,Claims,Patients,Avg_Claim,Billed,Paid
385,DR0386,Provider_0386,Family Medicine,1921,1790,238.96,"459,048.61","263,389.51"
34,DR0035,Provider_0035,Other,1552,1477,852.05,"1,322,384.35","757,414.83"
239,DR0240,Provider_0240,Pediatrics,899,868,222.10,"199,664.02","117,479.31"
241,DR0242,Provider_0242,General Surgery,823,802,"2,473.21","2,035,447.89","1,122,875.91"
80,DR0081,Provider_0081,Internal Medicine,761,751,256.41,"195,124.97","108,207.81"
111,DR0112,Provider_0112,Internal Medicine,740,727,248.10,"183,594.12","103,774.09"
371,DR0372,Provider_0372,Emergency Medicine,695,674,453.04,"314,864.14","178,165.04"
281,DR0282,Provider_0282,Family Medicine,672,653,232.26,"156,076.92","89,489.73"
208,DR0209,Provider_0209,Emergency Medicine,646,631,475.01,"306,854.57","175,156.08"
403,DR0404,Provider_0404,Orthopedics,626,611,"1,323.86","828,738.56","453,350.61"


## 7. Claim-value distribution and threshold selection

In [16]:
fact["Claim_Amount"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

count   75,025.00
mean       581.84
std      1,624.39
min          4.54
50%        183.24
75%        452.35
90%      1,145.70
95%      2,126.55
99%      7,664.40
max     85,000.00
Name: Claim_Amount, dtype: float64

In [17]:
p99 = fact["Claim_Amount"].quantile(0.99)

q1 = fact["Claim_Amount"].quantile(0.25)
q3 = fact["Claim_Amount"].quantile(0.75)
iqr_upper = q3 + 1.5 * (q3 - q1)

print("P99 threshold:", round(p99, 2))
print("IQR upper fence:", round(iqr_upper, 2))

P99 threshold: 7664.4
IQR upper fence: 1006.48


## 8. Anomaly rule: high-value claims

In [18]:
fact["High_Value_P99"] = (
    fact["Claim_Amount"] >= p99
).astype(int)

high_value_claims = fact[
    fact["High_Value_P99"] == 1
].sort_values("Claim_Amount", ascending=False)

high_value_claims[
    [
        "Claim_ID",
        "Provider_ID",
        "Specialty",
        "Payer",
        "Procedure_Name",
        "Claim_Amount",
    ]
].head(20)

,Claim_ID,Provider_ID,Specialty,Payer,Procedure_Name,Claim_Amount
24669,C00024670,DR0242,General Surgery,Self-Pay,Major Surgery,"85,000.00"
41567,C00041568,DR0242,General Surgery,Other,Major Surgery,"53,204.83"
34785,C00034786,DR0404,Orthopedics,Medicaid,Major Surgery,"53,135.54"
34110,C00034111,DR0218,General Surgery,Medicare,Major Surgery,"49,607.91"
22961,C00022962,DR0052,General Surgery,Commercial,Major Surgery,"46,749.23"
58039,C00058040,DR0256,General Surgery,Medicare Advantage,Major Surgery,"40,737.72"
17117,C00017118,DR0153,Orthopedics,Medicare,Major Surgery,"38,585.31"
69617,C00069618,DR0166,Cardiology,Commercial,Major Surgery,"34,934.64"
48496,C00048497,DR0311,Orthopedics,Commercial,Major Surgery,"33,479.28"
50961,C00050962,DR0242,General Surgery,Medicare Advantage,Major Surgery,"33,123.37"


## 9. Anomaly rule: provider weekly volume spikes

In [19]:
provider_weekly = (
    fact[fact["Provider_ID"].fillna("").ne("")]
    .groupby(["Provider_ID", "Week_Start"])
    .size()
    .reset_index(name="Weekly_Claims")
)

provider_baseline = (
    provider_weekly
    .groupby("Provider_ID")["Weekly_Claims"]
    .agg(["mean", "std"])
    .reset_index()
    .rename(columns={
        "mean": "Mean_Weekly_Claims",
        "std": "Std_Weekly_Claims"
    })
)

provider_weekly = provider_weekly.merge(
    provider_baseline,
    on="Provider_ID",
    how="left"
)

provider_weekly["Volume_Z"] = (
    provider_weekly["Weekly_Claims"]
    - provider_weekly["Mean_Weekly_Claims"]
) / provider_weekly["Std_Weekly_Claims"].replace(0, np.nan)

volume_spikes = provider_weekly[
    (provider_weekly["Volume_Z"] >= 3)
    & (provider_weekly["Weekly_Claims"] >= 8)
]

volume_spikes.sort_values("Volume_Z", ascending=False).head(20)

,Provider_ID,Week_Start,Weekly_Claims,Mean_Weekly_Claims,Std_Weekly_Claims,Volume_Z
15662,DR0191,2024-05-27,9,2.03,1.17,5.94
30802,DR0374,2025-08-25,13,3.50,1.85,5.13
11976,DR0148,2026-03-30,8,2.02,1.20,5.01
36659,DR0443,2026-03-30,11,3.16,1.72,4.56
23148,DR0281,2025-12-01,9,2.42,1.52,4.33
19601,DR0240,2024-12-30,18,6.24,2.72,4.32
6558,DR0081,2024-11-04,14,5.32,2.08,4.16
16346,DR0199,2025-02-24,8,2.43,1.34,4.15
24844,DR0301,2024-06-10,9,2.60,1.59,4.02
416,DR0006,2025-07-21,8,2.56,1.37,3.99


## 10. Anomaly rule: financial exceptions

In [20]:
financial_exceptions = fact[
    fact["Paid_Amount"] > fact["Allowed_Amount"]
]

financial_exceptions[
    ["Claim_ID", "Payer", "Claim_Amount", "Allowed_Amount", "Paid_Amount"]
].head(20)

,Claim_ID,Payer,Claim_Amount,Allowed_Amount,Paid_Amount
1121,C00001122,Commercial,299.45,265.31,275.69
4808,C00004809,Self-Pay,78.07,57.01,60.74
4877,C00004878,Commercial,601.27,515.08,566.31
5822,C00005823,Medicaid,61.10,55.56,59.11
5984,C00005985,Medicare Advantage,75.85,67.90,73.91
8370,C00008371,Commercial,52.44,48.21,50.21
11283,C00011284,Medicare,51.19,42.10,43.91
13028,C00013029,Self-Pay,65.59,51.97,53.93
17419,C00017420,Medicaid,56.95,46.67,48.94
20365,C00020366,Medicare,188.24,140.60,152.14


## 11. Combine operational screening rules

In [21]:
fact["Missing_Provider"] = (
    fact["Provider_ID"].fillna("").eq("")
).astype(int)

spike_keys = set(
    zip(
        volume_spikes["Provider_ID"],
        volume_spikes["Week_Start"]
    )
)

fact["Provider_Week_Spike"] = [
    int((provider, week) in spike_keys)
    for provider, week
    in zip(fact["Provider_ID"], fact["Week_Start"])
]

fact["Financial_Exception"] = (
    fact["Paid_Amount"] > fact["Allowed_Amount"]
).astype(int)

fact["Anomaly_Flag"] = (
    (fact["High_Value_P99"] == 1)
    | (fact["Financial_Exception"] == 1)
    | (fact["Missing_Provider"] == 1)
    | (fact["Provider_Week_Spike"] == 1)
).astype(int)

fact["Anomaly_Flag"].value_counts()

Anomaly_Flag
0    73646
1     1379
Name: count, dtype: int64

## 12. Evaluate against the separate synthetic ground truth

In [22]:
evaluation = (
    fact[["Claim_ID", "Anomaly_Flag"]]
    .drop_duplicates("Claim_ID")
    .merge(
        truth[["Claim_ID", "Ground_Truth_Flag"]].drop_duplicates("Claim_ID"),
        on="Claim_ID",
        how="left"
    )
)

evaluation["Ground_Truth_Flag"] = (
    evaluation["Ground_Truth_Flag"]
    .fillna(0)
    .astype(int)
)

In [23]:
true_positive = ((evaluation["Anomaly_Flag"] == 1) & (evaluation["Ground_Truth_Flag"] == 1)).sum()
false_positive = ((evaluation["Anomaly_Flag"] == 1) & (evaluation["Ground_Truth_Flag"] == 0)).sum()
false_negative = ((evaluation["Anomaly_Flag"] == 0) & (evaluation["Ground_Truth_Flag"] == 1)).sum()

precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0
recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

pd.DataFrame({
    "Metric": ["True Positives", "False Positives", "False Negatives", "Precision", "Recall", "F1"],
    "Value": [true_positive, false_positive, false_negative, precision, recall, f1],
})

,Metric,Value
0,True Positives,192.00
1,False Positives,"1,186.00"
2,False Negatives,142.00
3,Precision,0.14
4,Recall,0.57
5,F1,0.22


## 13. Export Power BI-ready files

In [24]:
OUTPUT_PATH.mkdir(exist_ok=True)

fact.to_csv(OUTPUT_PATH / "fact_claims_enriched.csv", index=False)
payer_summary.to_csv(OUTPUT_PATH / "payer_summary.csv", index=False)
provider_summary.to_csv(OUTPUT_PATH / "provider_summary.csv", index=False)

print("Export complete:", OUTPUT_PATH)

Export complete: /mnt/data/innovaccer_project2_claims_anomaly_project/05_outputs


## Interview takeaway

> I treated the project as an operational healthcare analytics workflow. I validated the raw claims in SQL, enriched them with patient/provider/facility/diagnosis/procedure dimensions, used Python for distribution analysis and explainable anomaly rules, and prepared Power BI-ready reporting. Each anomaly is a review signal rather than proof of fraud.